# Minimal 4-Oscillator Domain

Smallest possible SCPN Phase Orchestrator demo: 4 oscillators in 2 layers.
Shows that even a trivial binding spec produces meaningful coherence dynamics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scpn_phase_orchestrator.coupling.knm import CouplingBuilder
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

In [ ]:
N_OSC = 4
DT = 0.01
STEPS = 1500

omegas = np.array([1.0, 1.1, 0.95, 1.05])
coupling = CouplingBuilder().build(N_OSC, base_strength=0.45, decay_alpha=0.3)
alpha = coupling.alpha.copy()

rng = np.random.default_rng(42)
phases_init = rng.uniform(0, 2 * np.pi, N_OSC)

engine = UPDEEngine(N_OSC, DT)
phases = phases_init.copy()
hist = {"R_global": [], "R_lower": [], "R_upper": []}

LOWER = np.array([0, 1])
UPPER = np.array([2, 3])


def layer_R(phases, idx):
    return float(np.abs(np.mean(np.exp(1j * phases[idx]))))


for _ in range(STEPS):
    phases = engine.step(phases, omegas, coupling.knm, 0.0, 0.0, alpha)
    Rg, _ = compute_order_parameter(phases)
    hist["R_global"].append(Rg)
    hist["R_lower"].append(layer_R(phases, LOWER))
    hist["R_upper"].append(layer_R(phases, UPPER))

for k in hist:
    hist[k] = np.array(hist[k])

print(f"Final R_global: {hist['R_global'][-1]:.3f}")
print(f"Final R_lower:  {hist['R_lower'][-1]:.3f}")
print(f"Final R_upper:  {hist['R_upper'][-1]:.3f}")

In [ ]:
t = np.arange(STEPS) * DT

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, hist["R_lower"], color="#d62728", lw=1.2, label="R_lower")
ax.plot(t, hist["R_upper"], color="#1f77b4", lw=1.2, label="R_upper")
ax.plot(t, hist["R_global"], color="#2ca02c", lw=1.0, ls="--", alpha=0.7, label="R_global")
ax.set_xlabel("Time (s)")
ax.set_ylabel("R")
ax.set_ylim(-0.05, 1.05)
ax.set_title("Minimal Domain: 4 Oscillators, 2 Layers")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## Results

Even with only 4 oscillators and default coupling, the system converges to
near-unity coherence within a few seconds. This validates the minimal binding
spec at `domainpacks/minimal_domain/binding_spec.yaml`.